# Quickstart for Carsus

In this guide you will learn how to create atomic data files suitable for [TARDIS](https://github.com/tardis-sn/tardis).

<div class="alert alert-info">

**Note:**

Get familiar with the [Notation in Carsus](reference/notation.rst) and learn how to correctly select ions.
    
</div>

## Atomic Weights and Ionization Energies (NIST)

Download atomic weights and ionization energies from the National Institute of Standards and Technology (NIST).

In [1]:
from carsus.io.nist import NISTWeightsComp, NISTIonizationEnergies

/home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/vcs_versioning/_backends/_git.py:377: UserWarning: "/home/runner/work/carsus/carsus" is shallow and may cause errors
  pre_parse(wd)


In [2]:
atomic_weights = NISTWeightsComp()
ionization_energies = NISTIonizationEnergies('H-Zn')

[ carsus.io.nist.weightscomp][   INFO] - Downloading data from the carsus-dat-nist repository (weightscomp.py:77)


[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
 (_py_warnings.py:230)


[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")
 (_py_warnings.py:230)


[  carsus.io.nist.ionization][   INFO] - Downloading ionization energies from the carsus-data-nist repo. (ionization.py:91)


[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
 (_py_warnings.py:230)


## Levels, Lines, Collisions & Cross-sections

Carsus supports three sources of energy levels and spectral lines: **GFALL**, **CHIANTI** and **CMFGEN**.

### GFALL

The Robert Kurucz's Atomic Linelist (GFALL) reader is the main source of **levels and lines**.

<div class="alert alert-warning">

**Warning:**
    
Creating a `GFALLReader` instance is **required**.

</div>

In [3]:
!wget -qO /tmp/gfall.dat https://media.githubusercontent.com/media/tardis-sn/carsus-db/master/gfall/gfall_latest.dat

In [4]:
from carsus.io.kurucz import GFALLReader

gfall_reader = GFALLReader('H-Zn',
                           '/tmp/gfall.dat')

[     carsus.io.kurucz.gfall][WARNING] - A specific combination to identify unique levels from GFALL data has not been given. Defaulting to ["energy", "j"]. (gfall.py:122)


### CHIANTI

The Chianti Atomic Database reader provides levels and lines but also **collision strengths**.

<div class="alert alert-info">

**Note:**

Creating a `ChiantiReader` instance is **optional**. 

</div>

In [5]:
from carsus.io.chianti_ import ChiantiReader

chianti_reader = ChiantiReader('H-He', 
                               collisions=True, 
                               priority=20)

 ChiantiPy version 0.16.0 


 found PyQt5 widgets
 using CLI for selections
 reading chiantirc file


By default `priority` parameter is set to `10`. Increase this value if you want to keep CHIANTI levels and lines over GFALL.

### CMFGEN

The atomic database of [CMFGEN](https://sites.pitt.edu/~hillier/web/CMFGEN.htm) is a source of levels, lines and (optionally) **ionization energies**, **photoionization cross-sections** and **collisions**.

<div class="alert alert-info">
    
**Note:**

Creating a `CMFGENReader` instance is **optional**. 

</div>

<div class="alert alert-warning">

**Warning:**
    
Cross-sections require data from `H 0`, use this the reader with enough `priority` to select levels from this ion.

</div>

In [6]:
import pathlib

cmfgen_path = '../../carsus-data-cmfgen/atomic/'
if not pathlib.Path(cmfgen_path).exists():
    cmfgen_path = "/tmp/atomic/"

In [7]:
from carsus.io.cmfgen import CMFGENReader

cmfgen_reader = CMFGENReader.from_config('Si 0-1', 
                                         cmfgen_path, 
                                         priority=30,
                                         ionization_energies=True,
                                         cross_sections=True,
                                         collisions=False,
                                         temperature_grid=None,
                                         drop_mismatched_labels=True)

[      carsus.io.cmfgen.base][WARNING] - Selecting H 0 from CMFGEN (required to ingest cross-sections). (base.py:562)


[      carsus.io.cmfgen.base][   INFO] - Configuration schema found for H 0. (base.py:576)


[      carsus.io.cmfgen.base][   INFO] - Configuration schema found for Si 0. (base.py:576)


[      carsus.io.cmfgen.base][   INFO] - Configuration schema found for Si 1. (base.py:576)


[      carsus.io.cmfgen.base][   INFO] - Loading atomic data for H 0. (base.py:861)


[      carsus.io.cmfgen.base][   INFO] - Loading atomic data for Si 0. (base.py:861)


[      carsus.io.cmfgen.base][   INFO] - Loading atomic data for Si 1. (base.py:861)


## Zeta Data

Long & Knigge's ground state recombinations fractions ($\zeta$).

In [8]:
from carsus.io.zeta import KnoxLongZeta

zeta_data = KnoxLongZeta()

## Nuclear decay data

Carsus also supports the decay radiation data of all the nuclides at the [NNDC Archives](https://www.nndc.bnl.gov/ensdfarchivals/). The ENSDF data is stored in CSV format in the repository [carsus-data-nndc](https://github.com/tardis-sn/carsus-data-nndc).

### NNDC

The NNDCReader instance looks for the `carsus-data-nndc` repository in the local system at the path specified by the argument `dirname`. If the data is to be downloaded from the Github repository directly , the `remote` argument should be set to `True`.

<div class="alert alert-info">
    
**Note:**

Creating a `NNDCReader` instance is **optional**. 

</div>

In [9]:
from carsus.io.nuclear import NNDCReader

nndc_reader = NNDCReader(remote=True)

[     carsus.io.nuclear.nndc][WARNING] - Failed to clone the repository.
Check if the repository already exists at /home/runner/Downloads/carsus-data-nndc (nndc.py:44)


[     carsus.io.nuclear.nndc][   INFO] - Parsing decay data from: /home/runner/Downloads/carsus-data-nndc/csv (nndc.py:50)


fatal: destination path '/home/runner/Downloads/carsus-data-nndc' already exists and is not an empty directory.


## Create an Atomic Data File

Finally, create a `TARDISAtomData` object and dump the data with the `to_hdf` method.

In [10]:
from carsus.io.output import TARDISAtomData

atom_data = TARDISAtomData(atomic_weights,
                           ionization_energies,
                           gfall_reader,
                           zeta_data,
                           chianti_reader,
                           cmfgen_reader,
                           nndc_reader,
                           levels_lines_param={"lines_loggf_threshold": -3})  # Default cutoff is -3 (optional)

[carsus.io.output.levels_lines][   INFO] - Ingesting energy levels. (levels_lines.py:167)


[     carsus.io.kurucz.gfall][   INFO] - Parsing GFALL from: /tmp/gfall.dat (gfall.py:176)


[carsus.io.output.levels_lines][   INFO] - GFALL selected species: Li 0, Li 1, Be 0, Be 1, Be 2, B 0, B 1, B 2, B 3, C 0, C 1, C 2, C 3, N 0, N 1, N 2, N 3, N 4, N 5, O 0, O 1, O 2, O 3, O 4, O 5, F 0, F 1, F 2, F 3, F 4, F 5, Ne 0, Ne 1, Ne 2, Ne 3, Ne 4, Ne 5, Na 0, Na 1, Na 2, Na 3, Na 4, Na 5, Mg 0, Mg 1, Mg 2, Mg 3, Mg 4, Mg 5, Al 0, Al 1, Al 2, Al 3, Al 4, Al 5, Si 2, Si 3, Si 4, Si 5, P 0, P 1, P 2, P 3, P 4, P 5, S 0, S 1, S 2, S 3, S 4, S 5, Cl 0, Cl 1, Cl 2, Cl 3, Cl 4, Ar 0, Ar 1, Ar 2, Ar 3, Ar 4, K 0, K 1, K 2, K 3, K 4, Ca 0, Ca 1, Ca 2, Ca 3, Ca 4, Ca 5, Ca 6, Ca 7, Ca 8, Sc 0, Sc 1, Sc 2, Sc 3, Sc 4, Sc 5, Sc 6, Sc 7, Sc 8, Ti 0, Ti 1, Ti 2, Ti 3, Ti 4, Ti 5, Ti 6, Ti 7, Ti 8, V 0, V 1, V 2, V 3, V 4, V 5, V 6, V 7, V 8, Cr 0, Cr 1, Cr 2, Cr 3, Cr 4, Cr 5, Cr 6, Cr 7, Cr 8, Mn 0, Mn 1, Mn 2, Mn 3, Mn 4, Mn 5, Mn 6, Mn 7, Mn 8, Fe 0, Fe 1, Fe 2, Fe 3, Fe 4, Fe 5, Fe 6, Fe 7, Fe 8, Co 0, Co 1, Co 2, Co 3, Co 4, Co 5, Co 6, Co 7, Co 8, Ni 0, Ni 1, Ni 2, Ni 3, Ni 4, Ni 5, N

[carsus.io.output.levels_lines][   INFO] - Chianti selected species: He 0, He 1. (levels_lines.py:207)


[carsus.io.output.levels_lines][   INFO] - CMFGEN selected species: H 0, Si 0, Si 1. (levels_lines.py:211)


...

[carsus.io.output.levels_lines][   INFO] - Matching levels and lines. (levels_lines.py:338)


[                py.warnings][WARNING] - /home/runner/micromamba/envs/carsus/lib/python3.14/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
 (_py_warnings.py:230)


[carsus.io.output.collisions][   INFO] - Ingesting collisional strengths. (collisions.py:112)


[carsus.io.output.collisions][   INFO] - Matching collisions and levels. (collisions.py:124)


[      carsus.io.output.base][   INFO] - Finished. (base.py:98)


In [11]:
atom_data.to_hdf('kurucz_cd23_chianti_He_cmfgen_H_Si_I-II.h5')

[                py.warnings][WARNING] - /home/runner/work/carsus/carsus/carsus/io/output/base.py:217: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block7_values] [items->Index(['Element', 'Parent E(level)', 'Uncertainty', 'JPi',
       'Q Value Uncertainty', 'Gammas Balance', 'X-Rays Balance', 'B- Balance',
       'B+ Balance', 'Conversion Electrons Balance', 'Auger Electrons Balance',
       'Neutrinos Balance', 'Recoil Balance', 'Neutrons Balance',
       ' Protons Balance', 'Alphas Balance', 'Sum Balance',
       'Q-effective Balance', 'Missing Energy Balance', 'Radiation',
       'Rad subtype', 'Uncertainty.1', 'Uncertainty.3', 'Uncertainty.4'],
      dtype='str')]

  f.put(hdf_path, output)
 (_py_warnings.py:230)


[                py.warnings][WARNING] - /home/runner/work/carsus/carsus/carsus/io/output/base.py:217: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block2_values] [items->Index(['btemp', 'bscups'], dtype='str')]

  f.put(hdf_path, output)
 (_py_warnings.py:230)


[                py.warnings][WARNING] - /home/runner/work/carsus/carsus/carsus/io/output/base.py:217: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->values] [items->None]

  f.put(hdf_path, output)
 (_py_warnings.py:230)


[carsus.io.output.photo_ionization][   INFO] - Ingesting photoionization cross-sections. (photo_ionization.py:32)


[carsus.io.output.photo_ionization][   INFO] - Matching levels and cross sections. (photo_ionization.py:35)


[      carsus.io.output.base][   INFO] - Signing TARDISAtomData. (base.py:284)


[      carsus.io.output.base][   INFO] - Format Version: 2.0 (base.py:285)


[      carsus.io.output.base][   INFO] - MD5: 4d327a771c4eacd57ddb463fb2716c9d (base.py:286)


[      carsus.io.output.base][   INFO] - UUID1: c1f443127b0811f1a66f000d3a36b2d4 (base.py:287)


A lower value of log(g f) includes weaker transitions, while a higher value filters out more lines.

You are done! Now you can use your file to run TARDIS simulations.

### Metadata

Carsus stores metadata inside the HDF5 files to ensure reproducibility. This metadata includes a checksum for each stored table, version number or checksum of selected datasets, and versions of relevant packages. 

In [12]:
import pandas as pd

In [13]:
store = pd.HDFStore('kurucz_cd23_chianti_He_cmfgen_H_Si_I-II.h5', key='metadata')

In [14]:
store["metadata"]

value
field    key                                                    
format   version                                             2.0
md5sum   atom_data              3f0298f7de8c5a5015da18caa5863502
         collisions_data        2931d69509bb3e07de0349e516b4c0e8
         collisions_metadata    33a56c0abaec5ec37473c130e9e68ff6
         decay_radiation_data   567484a1aff18ccb8b8afce542a14f6c
         ionization_data        0e61c6350e896ddc7cc4c5d6a461fb50
         levels_data            0772a8ad23c99e4fa937162377df8cf5
         lines_data             96882df818b9147efb686ef68327b55d
         lines_metadata         f4f05f71aaf43cad228913bf5d5c5f4e
         macro_atom_data        24f467248f93ada1e4a8fee13f8d7027
         macro_atom_references  7d0c94a5f891e0670642e1a0f5c660cb
         photoionization_data   7e19e9ea0e1552414d374887e97627d6
         zeta_data              3555b5c9000ae58eb92ac5d0018ca360
datasets nist_weights                                        4.1
         nist_spectra                                       5.12
         gfall                  2704fbda0b8cba61bb70426234224464
         zeta                   a1d4bed2982e8d6a4f8b0076bf637e49
         chianti                                            10.0
         cmfgen                                       2016.11.15
software python                                           3.14.5
         carsus                              0.1.dev1+gba4eed8e1
         astropy                                           7.2.0
         numpy                                             2.4.6
         pandas                                            3.0.3
         tables                                           3.11.1
         ChiantiPy                                        0.16.0

In [15]:
store.root._v_attrs

/._v_attrs (AttributeSet), 8 attributes:
   [CLASS := np.str_('GROUP'),
    DATE := np.str_('2026-07-08T20:08:16.918103+00:00'),
    FORMAT_VERSION := np.str_('2.0'),
    MD5 := np.str_('4d327a771c4eacd57ddb463fb2716c9d'),
    PYTABLES_FORMAT_VERSION := np.str_('2.1'),
    TITLE := np.str_(''),
    UUID1 := np.str_('c1f443127b0811f1a66f000d3a36b2d4'),
    VERSION := np.str_('1.0')]